In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

# Загрузка данных

In [18]:
data = pd.read_excel('data_ford_price.xlsx') 

#  Отбор признаков: мотивация

In [ ]:
import pandas as pd

from geopy.geocoders import Nominatim

## Предобработка данных

In [3]:
data = data[['price','year', 'cylinders', 'odometer', 'lat', 'long', 'weather']]
data.dropna(inplace = True)

y = data['price']
x = data.drop(columns='price')

X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

## Обучение модели

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)
y_predicted = model.predict(X_test)
 
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

MAE: 4682.957


## Удаление избыточного признака

In [5]:
x.drop('lat', axis = 1, inplace = True)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

In [7]:
model = LinearRegression()
model.fit(X_train, y_train)
y_predicted = model.predict(X_test)
 
mae = mean_absolute_error(y_test, y_predicted)
print('MAE: %.3f' % mae)

MAE: 4672.930


#  Отбор признаков: классификация методов

## Метод рекурсивного исключения признаков

In [8]:
from sklearn.feature_selection import RFE

In [9]:
y = data['price']
x = data.drop(columns='price')

In [10]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.3, random_state=40)

In [11]:
estimator = LinearRegression()
selector = RFE(estimator, n_features_to_select=3, step=1)
selector = selector.fit(X_train, y_train)
 
selector.get_feature_names_out()

array(['year', 'cylinders', 'lat'], dtype=object)

In [12]:
X_train.columns

Index(['year', 'cylinders', 'odometer', 'lat', 'long', 'weather'], dtype='object')

In [13]:
selector.ranking_

array([1, 1, 4, 1, 3, 2])

##  МЕТОДЫ ВЫБОРА ПРИЗНАКОВ НА ОСНОВЕ ФИЛЬТРОВ

In [14]:
from sklearn.feature_selection import SelectKBest, f_regression

In [15]:
selector = SelectKBest(f_regression, k=3)
selector.fit(X_train, y_train)
 
selector.get_feature_names_out()

array(['year', 'cylinders', 'odometer'], dtype=object)

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.feature_selection import RFE, SelectKBest, f_regression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# загрузка данных
data = pd.read_excel('data_ford_price.xlsx')

# целевая переменная
y = data['price']
X = data.drop(columns=['price'])

# если есть категориальные признаки, преобразуем их в dummy-переменные
X = pd.get_dummies(X, drop_first=True)

# заполнение пропусков
X = X.fillna(X.median(numeric_only=True))

# разбиение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 1. Отбор признаков через RFE
rfe = RFE(estimator=LinearRegression(), n_features_to_select=3)
rfe.fit(X_train, y_train)

rfe_features = X_train.columns[rfe.support_]
print("RFE selected features:", list(rfe_features))

X_train_rfe = X_train[rfe_features]
X_test_rfe = X_test[rfe_features]

model_rfe = LinearRegression()
model_rfe.fit(X_train_rfe, y_train)
pred_rfe = model_rfe.predict(X_test_rfe)

rfe_r2 = r2_score(y_test, pred_rfe)
rfe_mse = mean_squared_error(y_test, pred_rfe)
rfe_mae = mean_absolute_error(y_test, pred_rfe)

print("\nRFE model quality:")
print("R2:", rfe_r2)
print("MSE:", rfe_mse)
print("MAE:", rfe_mae)

# 2. Отбор признаков через SelectKBest
skb = SelectKBest(score_func=f_regression, k=3)
skb.fit(X_train, y_train)

skb_features = X_train.columns[skb.get_support()]
print("\nSelectKBest selected features:", list(skb_features))

X_train_skb = X_train[skb_features]
X_test_skb = X_test[skb_features]

model_skb = LinearRegression()
model_skb.fit(X_train_skb, y_train)
pred_skb = model_skb.predict(X_test_skb)

skb_r2 = r2_score(y_test, pred_skb)
skb_mse = mean_squared_error(y_test, pred_skb)
skb_mae = mean_absolute_error(y_test, pred_skb)

print("\nSelectKBest model quality:")
print("R2:", skb_r2)
print("MSE:", skb_mse)
print("MAE:", skb_mae)

# 3. Сравнение результатов
if skb_r2 > rfe_r2:
    best_method = "SelectKBest"
else:
    best_method = "RFE"

print("\nBest method:", best_method)

RFE selected features: ['condition', 'drive_fwd', 'size_sub-compact']

RFE model quality:
R2: 0.1548846121398132
MSE: 160861567.58047655
MAE: 8072.499886937252

SelectKBest selected features: ['year', 'condition', 'odometer']

SelectKBest model quality:
R2: 0.4174296911999369
MSE: 110888021.26381831
MAE: 5187.81164748476

Best method: SelectKBest
